<a href="https://colab.research.google.com/github/lkipo/medIA-cmi/blob/master/segmentacion_retina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segmentación de vasos da retina cunha U-Net

Obxectivo: dada unha **fotografía de fondo de ollo**, marcar **que píxeles son vaso sanguíneo**.
É un problema real: a morfoloxía vascular da retina serve para seguir a retinopatía diabética, a
hipertensión ou o glaucoma.

Usamos **DRIVE** (*Digital Retinal Images for Vessel Extraction*): 40 imaxes de 565×584 píxeles con
máscaras manuais de vasos (20 para adestrar e 20 para validar).

**Deseñado para Colab:** é un dataset minúsculo (~30 MB), usamos só o canal verde, adestramos con
recortes pequenos e unha U-Net de ~0.5 M de parámetros. Con GPU (T4) o adestramento leva ~1–2 minutos. **Sen GPU é moito máis lento** (decenas de minutos): activa a GPU, ou baixa `epocas` a 5–6 para unha proba rápida.

> En Colab: *Entorno de execución → Cambiar tipo → GPU (T4)*.

**Esquema:**
1. Datos e preprocesado
2. U-Net (con saltos)
3. Perda Dice + BCE
4. Adestramento con recortes aleatorios
5. Avaliación: Dice, IoU, sensibilidade, especificidade
6. Análise de erros

## Instalación de dependencias

Comproba que estean todas as bibliotecas necesarias e instala automaticamente as que falten (en Colab xa vén case todo instalado).

In [ ]:
import importlib.util, subprocess, sys

# (nome para importar, nome do paquete en pip)
REQUISITOS = [("numpy", "numpy"), ("matplotlib", "matplotlib"), ("PIL", "pillow"), ("scipy", "scipy"), ("torch", "torch")]

faltan = [pip for mod, pip in REQUISITOS if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", faltan)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
else:
    print("Todas as dependencias están instaladas.")

## 0. Preparación

In [ ]:
import os, time, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| dispositivo:", device)

## 1. Descargar os datos

Descargamos as 40 imaxes (e as súas máscaras) dun espello en Hugging Face. Só ocorre unha vez.

In [ ]:
BASE = "https://huggingface.co/datasets/Zomba/DRIVE-digital-retinal-images-for-vessel-extraction/resolve/main"
ROOT = "drive_data"
# no dataset orixinal: adestramento = imaxes 21-40, test = 01-20 (aquí chamado «val»)
SPLITS = {"train": range(21, 41), "val": range(1, 21)}

def nome(split, sub, i):
    # as máscaras de «val» chámanse 01_manual1.png, as de «train» 21.png
    if sub == "input":
        return f"{i:02d}.tif"
    return f"{i:02d}_manual1.png" if split == "val" else f"{i:02d}.png"

for split, ids in SPLITS.items():
    for sub in ["input", "label"]:
        os.makedirs(f"{ROOT}/{split}/{sub}", exist_ok=True)
        for i in ids:
            f = f"{ROOT}/{split}/{sub}/{nome(split, sub, i)}"
            if not os.path.exists(f):
                urllib.request.urlretrieve(f"{BASE}/{split}/{sub}/{nome(split, sub, i)}", f)
print("Descarga completa:", {s: len(os.listdir(f"{ROOT}/{s}/input")) for s in SPLITS})

## 2. Preprocesado

Cada imaxe pasa por catro pasos:

1. **Canal verde**: é o que mellor contraste dá aos vasos (o vermello satura e o azul é ruidoso).
2. **Máscara do campo de visión (FOV)**: o círculo da retina; fóra é negro e non nos interesa.
   Calculámola por limiar.
3. **Normalización**: restamos a media e dividimos pola desviación típica **dentro do FOV**, para que o
   modelo non dependa da iluminación de cada foto.
4. **Tensores**: gardamos todo en memoria (as 40 imaxes ocupan <100 MB).

In [ ]:
def cargar(split):
    imgs, labs, fovs = [], [], []
    for i in SPLITS[split]:
        rgb = np.array(Image.open(f"{ROOT}/{split}/input/{i:02d}.tif").convert("RGB"))
        lab = np.array(Image.open(f"{ROOT}/{split}/label/{nome(split, 'label', i)}").convert("L")) > 127

        verde = rgb[..., 1].astype(np.float32)
        # FOV: todo o que non é fondo negro, limpando buratos e bordos
        fov = rgb.max(axis=2) > 20
        fov = ndimage.binary_fill_holes(ndimage.binary_opening(fov, iterations=3))
        fov = ndimage.binary_erosion(fov, iterations=4)

        m, s = verde[fov].mean(), verde[fov].std()
        verde = (verde - m) / s
        verde[~fov] = 0                       # fóra do FOV = 0 (valor medio)

        imgs.append(verde); labs.append(lab & fov); fovs.append(fov)
    t = lambda a, dt: torch.tensor(np.stack(a)).unsqueeze(1).to(dt)
    return t(imgs, torch.float32), t(labs, torch.float32), t(fovs, torch.bool), None

x_tr, y_tr, f_tr, _ = cargar("train")
x_va, y_va, f_va, _ = cargar("val")
print("train:", tuple(x_tr.shape), "| val:", tuple(x_va.shape))
print(f"Fracción de píxeles de vaso (dentro do FOV): {y_tr.sum().item() / f_tr.sum().item():.1%}")

### Mirar os datos

Fíxate na proporción: **só ~13 % dos píxeles son vaso**. É o clásico **desbalance de clases** da
segmentación médica, e condiciona a elección da perda e das métricas.

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for k in range(2):
    rgb = np.array(Image.open(f"{ROOT}/train/input/{21+k:02d}.tif").convert("RGB"))
    ax[k, 0].imshow(rgb);                       ax[k, 0].set_title("RGB orixinal")
    ax[k, 1].imshow(rgb[..., 1], cmap="gray");  ax[k, 1].set_title("canal verde")
    ax[k, 2].imshow(x_tr[k, 0].cpu(), cmap="gray"); ax[k, 2].set_title("normalizada")
    ax[k, 3].imshow(y_tr[k, 0].cpu(), cmap="gray"); ax[k, 3].set_title("máscara de vasos")
for a in ax.flat: a.axis("off")
plt.tight_layout(); plt.show()

## 3. A U-Net

Unha **U-Net** é unha rede codificador–decodificador:

- O **codificador** (esquerda) aplica convolucións e *pooling*: a imaxe fai-se máis pequena pero
  captura contexto ( «isto parece unha ramificación»).
- O **decodificador** (dereita) volve subir á resolución orixinal con convolucións transpostas.
- As **conexións de salto** (*skip connections*) copian os mapas do codificador ao decodificador do
  mesmo nivel. Devolven o detalle fino que o *pooling* destrúe (precisamente o que necesitan uns vasos de
  1–3 píxeles de ancho).

A saída ten **1 canle**: un *logit* por píxel (probabilidade de ser vaso despois dun sigmoide).

```
entrada 1×H×W ─► [C16] ───────────────────────────► concat ─► [C16] ─► 1×H×W
                    ↓ pool                              ↑ up
                  [C32] ─────────────────────► concat ─► [C32]
                    ↓ pool                        ↑ up
                  [C64] ─────────────► concat ─► [C64]
                    ↓ pool                  ↑ up
                              [C128]  (fondo)
```

In [ ]:
class Bloque(nn.Module):
    '''Dúas convolucións 3x3 + BatchNorm + ReLU.'''
    def __init__(self, c_in, c_out):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.f(x)


class UNet(nn.Module):
    def __init__(self, c_in=1, c_out=1, base=16, skip=True):
        super().__init__()
        self.skip = skip
        b = base
        self.e1, self.e2, self.e3 = Bloque(c_in, b), Bloque(b, 2*b), Bloque(2*b, 4*b)
        self.fondo = Bloque(4*b, 8*b)
        self.u3 = nn.ConvTranspose2d(8*b, 4*b, 2, stride=2)
        self.u2 = nn.ConvTranspose2d(4*b, 2*b, 2, stride=2)
        self.u1 = nn.ConvTranspose2d(2*b, b, 2, stride=2)
        m = 2 if skip else 1                       # con saltos, o concat duplica as canles
        self.d3, self.d2, self.d1 = Bloque(m*4*b, 4*b), Bloque(m*2*b, 2*b), Bloque(m*b, b)
        self.saida = nn.Conv2d(b, c_out, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        f = self.fondo(F.max_pool2d(e3, 2))
        cat = (lambda a, b: torch.cat([a, b], 1)) if self.skip else (lambda a, b: a)
        d3 = self.d3(cat(self.u3(f), e3))
        d2 = self.d2(cat(self.u2(d3), e2))
        d1 = self.d1(cat(self.u1(d2), e1))
        return self.saida(d1)                      # logits, sen sigmoide


modelo = UNet().to(device)
print("Parámetros:", f"{sum(p.numel() for p in modelo.parameters()):,}")
print("Forma da saída para unha entrada 1x1x128x128:", tuple(modelo(torch.zeros(1, 1, 128, 128, device=device)).shape))

## 4. Perda e métricas

Con só ~13 % de píxeles positivos, un modelo que dixese «nada é vaso» tería un **87 % de accuracy** e
sería inútil. Por iso:

- **Dice** (e **IoU**) miden o solapamento entre predición $P$ e verdade $G$:

$$ \text{Dice} = \frac{2\,|P\cap G|}{|P|+|G|}, \qquad \text{IoU} = \frac{|P\cap G|}{|P\cup G|} $$

- Adestramos con **BCE + Dice loss**: a BCE (entropía cruzada binaria) dá gradientes estables
  píxel a píxel, e a Dice loss ($1-\text{Dice}$ diferenciable) céntrase no solapamento e compensa o
  desbalance.

In [ ]:
def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2, 3))
    union = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return (1 - (2 * inter + eps) / (union + eps)).mean()

def perda_total(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target) + dice_loss(logits, target)


@torch.no_grad()
def metricas(pred_bin, target, fov):
    '''pred_bin, target, fov: tensores booleanos (N,1,H,W). Só se contan píxeles dentro do FOV.'''
    p, g = pred_bin & fov, target.bool() & fov
    tp = (p & g).sum().item()
    fp = (p & ~g & fov).sum().item()
    fn = (~p & g).sum().item()
    tn = (~p & ~g & fov).sum().item()
    return {
        "accuracy": (tp + tn) / (tp + tn + fp + fn),
        "dice": 2 * tp / (2 * tp + fp + fn),
        "iou": tp / (tp + fp + fn),
        "sensibilidade": tp / (tp + fn),       # vasos atopados
        "especificidade": tn / (tn + fp),      # fondo ben descartado
    }

# Referencia absurda: «nada é vaso»
print("Modelo trivial (todo fondo):", {k: round(v, 3) for k, v in metricas(torch.zeros_like(y_va).bool(), y_va, f_va).items()})

## 5. Adestramento con recortes aleatorios

Meter imaxes completas de 584×565 gasta moita memoria. En vez diso, en cada iteración:

- collemos **recortes aleatorios de 128×128** (lote de 16),
- aplicamos **aumento de datos**: xiros e voltas aleatorias. Con só 20 imaxes isto é esencial para non
  memorizar.

Para **validar** si usamos a imaxe completa (a rede é totalmente convolucional, así que acepta calquera
tamaño múltiplo de 8; engadimos *padding*).

In [ ]:
CROP, BATCH = 128, 16

def lote(x, y, f=None):
    '''Recortes aleatorios + voltas/xiros de 90 graos.'''
    N, _, H, W = x.shape
    xs, ys = [], []
    for _ in range(BATCH):
        n = np.random.randint(N)
        i, j = np.random.randint(H - CROP + 1), np.random.randint(W - CROP + 1)
        a, b = x[n, :, i:i+CROP, j:j+CROP], y[n, :, i:i+CROP, j:j+CROP]
        k = np.random.randint(4)
        a, b = torch.rot90(a, k, (1, 2)), torch.rot90(b, k, (1, 2))
        if np.random.rand() < 0.5:
            a, b = a.flip(2), b.flip(2)
        xs.append(a); ys.append(b)
    return torch.stack(xs), torch.stack(ys)


@torch.no_grad()
def predecir(modelo, x):
    '''Inferencia sobre imaxes completas, con padding a múltiplo de 8.'''
    modelo.eval()
    H, W = x.shape[-2:]
    ph, pw = (-H) % 8, (-W) % 8
    out = []
    for i in range(0, len(x), 4):
        xb = F.pad(x[i:i+4], (0, pw, 0, ph), mode="reflect")
        out.append(torch.sigmoid(modelo(xb))[..., :H, :W])
    return torch.cat(out)


def adestrar(modelo, epocas=30, lotes_por_epoca=50, lr=2e-3, avaliar_cada=5):
    '''Unha época = `lotes_por_epoca` lotes de recortes aleatorios (16 x 50 = 800 recortes).'''
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epocas * lotes_por_epoca)
    hist = {"epoca": [], "perda": [], "dice_val": []}
    t0 = time.time()
    perdas = []

    for ep in range(1, epocas + 1):
        modelo.train()
        suma = 0.0
        for _ in range(lotes_por_epoca):
            xb, yb = lote(x_tr, y_tr)
            perda = perda_total(modelo(xb), yb)
            opt.zero_grad(); perda.backward(); opt.step(); sched.step()
            suma += perda.item()
        perdas.append(suma / lotes_por_epoca)

        if ep % avaliar_cada == 0 or ep == epocas:
            prob = predecir(modelo, x_va)
            m = metricas(prob > 0.5, y_va, f_va)
            hist["epoca"].append(ep); hist["perda"].append(perdas[-1]); hist["dice_val"].append(m["dice"])
            print(f"época {ep:3d}/{epocas} | perda {perdas[-1]:.3f} | Dice val {m['dice']:.3f} | {time.time()-t0:.0f}s")
    return hist

hist = adestrar(modelo)

> **Tempo:** con GPU T4 ~1–2 min. En CPU cada lote tarda ~1–2 s, así que as 30 épocas (1500 lotes) son moitos minutos:
> baixa `epocas` se non tes GPU (cun adestramento moi curto xa se obtén un Dice de ~0.7).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist["epoca"], hist["perda"], "o-"); ax[0].set_title("Perda de adestramento (BCE + Dice)")
ax[1].plot(hist["epoca"], hist["dice_val"], "o-", c="C1"); ax[1].set_title("Dice de validación")
for a in ax: a.set_xlabel("época"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Avaliación nas 20 imaxes de validación

In [ ]:
prob_va = predecir(modelo, x_va)
res = metricas(prob_va > 0.5, y_va, f_va)
for k, v in res.items():
    print(f"{k:15s}: {v:.3f}")

Compara co modelo trivial: a *accuracy* (0.87) sobe pouco, pero o **Dice pasa de 0 a un valor moito maior**.
A accuracy é enganosa con clases desbalanceadas; Dice e IoU non.


### Visualización

In [ ]:
def mostrar(idxs):
    fig, ax = plt.subplots(len(idxs), 4, figsize=(16, 4 * len(idxs)))
    ax = np.atleast_2d(ax)
    for r, n in enumerate(idxs):
        img = x_va[n, 0].cpu().numpy()
        g = y_va[n, 0].bool().cpu().numpy()
        p = (prob_va[n, 0] > 0.5).cpu().numpy() & f_va[n, 0].cpu().numpy()
        # superposición: verde = acerto, vermello = falso positivo, azul = falso negativo
        ov = np.zeros(g.shape + (3,)); ov[p & g] = [0, 1, 0]; ov[p & ~g] = [1, 0, 0]; ov[~p & g] = [0, 0.4, 1]
        d = 2 * (p & g).sum() / (p.sum() + g.sum())
        ax[r, 0].imshow(img, cmap="gray");  ax[r, 0].set_title(f"imaxe val {n+1}")
        ax[r, 1].imshow(g, cmap="gray");     ax[r, 1].set_title("verdade (observador 1)")
        ax[r, 2].imshow(prob_va[n, 0].cpu(), cmap="magma"); ax[r, 2].set_title("probabilidade predita")
        ax[r, 3].imshow(ov);                 ax[r, 3].set_title(f"Dice {d:.3f}  (verde ok, vermello FP, azul FN)")
    for a in ax.flat: a.axis("off")
    plt.tight_layout(); plt.show()

mostrar([0, 1, 2])

## 7. Análise de erros

Non todas as imaxes son igual de fáciles. Miremos o Dice **por imaxe** e a peor.

In [ ]:
dices = []
for n in range(len(x_va)):
    m = metricas((prob_va[n:n+1] > 0.5), y_va[n:n+1], f_va[n:n+1])
    dices.append(m["dice"])
dices = np.array(dices)

plt.figure(figsize=(9, 3.5))
plt.bar(range(1, len(dices) + 1), dices)
plt.axhline(dices.mean(), color="r", ls="--", label=f"media {dices.mean():.3f}")
plt.xlabel("imaxe de validación"); plt.ylabel("Dice"); plt.ylim(0.5, 0.95); plt.legend(); plt.grid(alpha=0.3)
plt.show()

peor = int(dices.argmin())
print(f"Peor imaxe: {peor+1} (Dice {dices[peor]:.3f})")
mostrar([peor])

**Preguntas para discutir**

- Onde falla a rede? (vasos moi finos, bordo do FOV, lesións que se parecen a vasos…)
- Os **falsos negativos** (azul) concéntranse nos capilares finos. É grave segundo a aplicación clínica?
- Podemos axustar o limiar 0.5? Probalo na seguinte cela: como cambian sensibilidade e especificidade?

In [ ]:
print("limiar | Dice | sensib. | especif.")
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    m = metricas(prob_va > t, y_va, f_va)
    print(f" {t:.1f}   | {m['dice']:.3f} | {m['sensibilidade']:.3f}   | {m['especificidade']:.3f}")

## 8. Experimento: son útiles as conexións de salto?

Adestramos a mesma rede **sen** *skip connections* (un simple codificador–decodificador) co mesmo
orzamento. Con adestramento completo debería perder detalle fino (con poucas épocas a diferenza pode non verse).

In [ ]:
torch.manual_seed(0); np.random.seed(0)
sen_skip = UNet(skip=False).to(device)
hist_ns = adestrar(sen_skip, epocas=30, avaliar_cada=10)

m_ns = metricas(predecir(sen_skip, x_va) > 0.5, y_va, f_va)
print(f"\nDice con saltos: {res['dice']:.3f}  |  sen saltos: {m_ns['dice']:.3f}")

## 9. Resumo e ideas para probar

**Que fixemos:** preprocesamos imaxes de retina, construímos unha U-Net, adestrámola con recortes
aleatorios e augmentación, e avaliámola con métricas apropiadas ao desbalance de clases.

**Para explorar:**

1. Cambia `base=16` a `8` ou `32`. Como cambian parámetros, tempo e Dice?
2. Perda só BCE ou só Dice. Que pasa coa sensibilidade?
3. Quita o aumento de datos (`rot90` e `flip`). Hai sobreaxuste (Dice de adestramento ≫ validación)?
4. Usa as tres canles RGB (`c_in=3`) en vez de só o verde.
5. **Cambio de dominio:** que pasaría se aplicases este modelo a imaxes doutra cámara ou hospital?
6. Compara con **Swin UNETR** (3D, no seguinte bloque do curso): mesmos conceptos —codificador,
   decodificador, saltos, Dice— pero en volumes e con *transformers*.